## Data Integration

Combining data from multiple sources to create comprehensive, unified datasets for analysis.

#### 1. Verify Data Formats and Key Columns

Before integration, we need to check the structure of each dataset and identify common keys.

In [ ]:
# Check key columns in each dataset for integration
print("Dataset Structures for Integration\n")

print("players_df columns:", players_df.columns.tolist()[:10], "...")
print("  Key: playerID")
print("  Rows:", len(players_df))

print("\nplayers_teams_df columns:", players_teams_df.columns.tolist()[:10], "...")
print("  Keys: playerID, year, tmID")
print("  Rows:", len(players_teams_df))

print("\nawards_players_df columns:", awards_players_df.columns.tolist())
print("  Keys: playerID, year")
print("  Rows:", len(awards_players_df))

print("\nteams_df columns:", teams_df.columns.tolist()[:10], "...")
print("  Keys: year, tmID")
print("  Rows:", len(teams_df))

print("\nteams_post_df columns:", teams_post_df.columns.tolist())
print("  Keys: year, tmID")
print("  Rows:", len(teams_post_df))

print("\ncoaches_df columns:", coaches_df.columns.tolist())
print("  Keys: coachID, year, tmID")
print("  Rows:", len(coaches_df))

print("\nseries_post_df columns:", series_post_df.columns.tolist())
print("  Keys: year, round, tmIDWinner, tmIDLoser")
print("  Rows:", len(series_post_df))

#### 2. Standardize Data Formats

Ensure consistent data types and formats across datasets for successful integration.

In [ ]:
# Standardize year formats (ensure all are integers)
for df_name, df in [('players_teams_df', players_teams_df), 
                     ('awards_players_df', awards_players_df),
                     ('teams_df', teams_df), 
                     ('teams_post_df', teams_post_df),
                     ('coaches_df', coaches_df),
                     ('series_post_df', series_post_df)]:
    if 'year' in df.columns:
        df['year'] = df['year'].astype(int)
        print(f"{df_name}: year converted to int")

# Standardize team IDs (ensure strings and strip whitespace)
for df_name, df in [('players_teams_df', players_teams_df),
                     ('teams_df', teams_df),
                     ('teams_post_df', teams_post_df),
                     ('coaches_df', coaches_df)]:
    if 'tmID' in df.columns:
        df['tmID'] = df['tmID'].astype(str).str.strip()
        print(f"{df_name}: tmID standardized")

# Standardize team IDs in series_post_df
series_post_df['tmIDWinner'] = series_post_df['tmIDWinner'].astype(str).str.strip()
series_post_df['tmIDLoser'] = series_post_df['tmIDLoser'].astype(str).str.strip()
print("series_post_df: tmIDWinner and tmIDLoser standardized")

# Standardize player IDs
for df_name, df in [('players_teams_df', players_teams_df),
                     ('awards_players_df', awards_players_df)]:
    if 'playerID' in df.columns:
        df['playerID'] = df['playerID'].astype(str).str.strip()
        print(f"{df_name}: playerID standardized")

# Standardize bioID in players_df
if 'bioID' in players_df.columns:
    players_df['bioID'] = players_df['bioID'].astype(str).str.strip()
    print("players_df: bioID standardized")

# Standardize coach IDs
coaches_df['coachID'] = coaches_df['coachID'].astype(str).str.strip()
print("coaches_df: coachID standardized")

print("\n=== Data Format Standardization Complete ===")

#### 3. Create Integrated Player Dataset

Merge player information with their team performance and awards.

In [ ]:
# Merge players with their team statistics
# Note: players_df uses 'bioID' while players_teams_df uses 'playerID'
# We'll join players_teams with basic player attributes

players_integrated = players_teams_df.copy()

print(f"Starting with players_teams data: {len(players_integrated)} records")
print(f"  Columns: {players_integrated.shape[1]}")

# Add awards information (count awards per player per year)
awards_summary = awards_players_df.groupby(['playerID', 'year']).agg({
    'award': 'count'
}).rename(columns={'award': 'num_awards'}).reset_index()

players_integrated = players_integrated.merge(
    awards_summary,
    on=['playerID', 'year'],
    how='left'
)

# Fill NaN awards with 0 (players with no awards)
players_integrated['num_awards'] = players_integrated['num_awards'].fillna(0).astype(int)

print(f"Added awards information")
print(f"  Players with awards in dataset: {(players_integrated['num_awards'] > 0).sum()}")

# Add team performance data
team_performance_cols = ['year', 'tmID', 'playoff', 'won', 'lost', 'confID', 'GP', 'name']
players_integrated = players_integrated.merge(
    teams_df[team_performance_cols],
    on=['year', 'tmID'],
    how='left',
    suffixes=('', '_team')
)

print(f"Added team performance data")

# Display sample
print("\n=== Integrated Player Dataset Sample ===")
display(players_integrated.head())
print(f"\nTotal records: {len(players_integrated)}")
print(f"Total columns: {players_integrated.shape[1]}")

#### 4. Create Integrated Team Dataset

Merge team regular season data with playoff performance and coaching information.

In [ ]:
# Start with teams regular season data
teams_integrated = teams_df.copy()

print(f"Starting with teams_df: {len(teams_integrated)} records")

# Add playoff performance
playoff_cols = ['year', 'tmID', 'W', 'L']
teams_integrated = teams_integrated.merge(
    teams_post_df[playoff_cols],
    on=['year', 'tmID'],
    how='left',
    suffixes=('', '_playoff')
)

# Rename playoff columns for clarity
teams_integrated.rename(columns={'W': 'playoff_wins', 'L': 'playoff_losses'}, inplace=True)

print(f"Added playoff data")
print(f"  Teams with playoff appearances: {teams_integrated['playoff_wins'].notna().sum()}")

# Add coach information (aggregate if multiple coaches per team-year)
coaches_agg = coaches_df.groupby(['year', 'tmID']).agg({
    'coachID': lambda x: ', '.join(x),  # Combine multiple coaches
    'won': 'sum',
    'lost': 'sum',
    'post_wins': 'sum',
    'post_losses': 'sum'
}).reset_index()

coaches_agg.rename(columns={
    'coachID': 'coaches',
    'won': 'coach_won',
    'lost': 'coach_lost',
    'post_wins': 'coach_post_wins',
    'post_losses': 'coach_post_losses'
}, inplace=True)

teams_integrated = teams_integrated.merge(
    coaches_agg,
    on=['year', 'tmID'],
    how='left'
)

print(f"Added coaching data")
print(f"  Teams with coach info: {teams_integrated['coaches'].notna().sum()}")

# Add championship information from series_post
champions = series_post_df[series_post_df['round'] == 'F'][['year', 'tmIDWinner']].copy()
champions['champion'] = 1
champions.rename(columns={'tmIDWinner': 'tmID'}, inplace=True)

teams_integrated = teams_integrated.merge(
    champions,
    on=['year', 'tmID'],
    how='left'
)

teams_integrated['champion'] = teams_integrated['champion'].fillna(0).astype(int)

print(f"Added championship flag")
print(f"  Championship teams: {teams_integrated['champion'].sum()}")

# Display sample
print("\n=== Integrated Team Dataset Sample ===")
display(teams_integrated.head())
print(f"\nTotal records: {len(teams_integrated)}")
print(f"Total columns: {teams_integrated.shape[1]}")

#### 5. Create Coach Performance Dataset

Integrate coach data with team performance to analyze coaching effectiveness.

In [ ]:
# Merge coaches with team performance
coaches_integrated = coaches_df.merge(
    teams_df[['year', 'tmID', 'name', 'confID', 'playoff', 'GP', 'arena', 'attend']],
    on=['year', 'tmID'],
    how='left'
)

print(f"Merged coaches with team info: {len(coaches_integrated)} records")

# Add playoff results for coaches
coaches_integrated = coaches_integrated.merge(
    teams_post_df[['year', 'tmID', 'W', 'L']],
    on=['year', 'tmID'],
    how='left',
    suffixes=('', '_team_playoff')
)

coaches_integrated.rename(columns={'W': 'team_playoff_W', 'L': 'team_playoff_L'}, inplace=True)

print(f"Added team playoff results")

# Add championship flag
champions = series_post_df[series_post_df['round'] == 'F'][['year', 'tmIDWinner']].copy()
champions['champion'] = 1
champions.rename(columns={'tmIDWinner': 'tmID'}, inplace=True)

coaches_integrated = coaches_integrated.merge(
    champions[['year', 'tmID', 'champion']],
    on=['year', 'tmID'],
    how='left'
)

coaches_integrated['champion'] = coaches_integrated['champion'].fillna(0).astype(int)

print(f"Added championship information")
print(f"  Coaches who won championships: {coaches_integrated['champion'].sum()}")

# Calculate win percentage
coaches_integrated['win_pct'] = coaches_integrated['won'] / (coaches_integrated['won'] + coaches_integrated['lost']) * 100

# Display sample
print("\n=== Integrated Coach Dataset Sample ===")
display(coaches_integrated.head())
print(f"\nTotal records: {len(coaches_integrated)}")
print(f"Total columns: {coaches_integrated.shape[1]}")

#### 6. Create Award Winners Dataset

Integrate award information with player and team data for comprehensive analysis.

In [ ]:
# Merge awards with player statistics
awards_integrated = awards_players_df.copy()

print(f"Starting with awards data: {len(awards_integrated)} records")

# Add player statistics for the award year
player_stats_cols = ['playerID', 'year', 'tmID', 'GP', 'points', 'rebounds', 'assists', 
                     'steals', 'blocks', 'turnovers']
awards_integrated = awards_integrated.merge(
    players_teams_df[player_stats_cols],
    on=['playerID', 'year'],
    how='left'
)

print(f"Added player statistics for award years")

# Add team information
team_info_cols = ['year', 'tmID', 'name', 'playoff', 'won', 'lost']
awards_integrated = awards_integrated.merge(
    teams_df[team_info_cols],
    on=['year', 'tmID'],
    how='left'
)

print(f"Added team information")

# Add championship flag
awards_integrated = awards_integrated.merge(
    champions[['year', 'tmID', 'champion']],
    on=['year', 'tmID'],
    how='left'
)

awards_integrated['champion'] = awards_integrated['champion'].fillna(0).astype(int)

print(f"Added championship information")
print(f"  Awards won by players on championship teams: {awards_integrated['champion'].sum()}")

# Display sample
print("\n=== Integrated Awards Dataset Sample ===")
display(awards_integrated.head(10))
print(f"\nTotal records: {len(awards_integrated)}")
print(f"Total columns: {awards_integrated.shape[1]}")
print(f"\nAward distribution:")
display(awards_integrated['award'].value_counts())

#### 7. Validate Integration Quality

Check for data quality issues after integration.

In [ ]:
print("=== Data Integration Quality Report ===\n")

# Check players_integrated
print("1. PLAYERS INTEGRATED DATASET")
print(f"   Total records: {len(players_integrated)}")
print(f"   Missing values by column:")
missing_players = players_integrated.isnull().sum()
display(missing_players[missing_players > 0])
print(f"   Duplicate records: {players_integrated.duplicated().sum()}")

# Check teams_integrated
print("\n2. TEAMS INTEGRATED DATASET")
print(f"   Total records: {len(teams_integrated)}")
print(f"   Missing values by column:")
missing_teams = teams_integrated.isnull().sum()
display(missing_teams[missing_teams > 0])
print(f"   Duplicate records: {teams_integrated.duplicated().sum()}")
print(f"   Teams with playoff data: {teams_integrated['playoff_wins'].notna().sum()}")

# Check coaches_integrated
print("\n3. COACHES INTEGRATED DATASET")
print(f"   Total records: {len(coaches_integrated)}")
print(f"   Missing values by column:")
missing_coaches = coaches_integrated.isnull().sum()
display(missing_coaches[missing_coaches > 0])
print(f"   Duplicate records: {coaches_integrated.duplicated().sum()}")

# Check awards_integrated
print("\n4. AWARDS INTEGRATED DATASET")
print(f"   Total records: {len(awards_integrated)}")
print(f"   Missing values by column:")
missing_awards = awards_integrated.isnull().sum()
display(missing_awards[missing_awards > 0])
print(f"   Duplicate records: {awards_integrated.duplicated().sum()}")

# Entity matching verification
print("\n5. ENTITY MATCHING VERIFICATION")
print(f"   Unique players in players_integrated: {players_integrated['playerID'].nunique()}")
print(f"   Unique teams in teams_integrated: {teams_integrated['tmID'].nunique()}")
print(f"   Unique coaches in coaches_integrated: {coaches_integrated['coachID'].nunique()}")
print(f"   Year range in all datasets: {players_integrated['year'].min()}-{players_integrated['year'].max()}")

print("\nIntegration quality check complete!")

#### 8. Summary of Integrated Datasets

The integrated datasets are now ready for advanced analysis and modeling.

In [ ]:
print("=== INTEGRATED DATASETS SUMMARY ===\n")

datasets_info = {
    'players_integrated': {
        'dataset': players_integrated,
        'description': 'Player stats merged with personal info, awards, and team performance',
        'key_use_cases': [
            'Analyze player performance across teams',
            'Correlate awards with team success',
            'Study player statistics over time'
        ]
    },
    'teams_integrated': {
        'dataset': teams_integrated,
        'description': 'Team regular season data merged with playoff performance, coaching, and championships',
        'key_use_cases': [
            'Analyze team performance trends',
            'Study playoff success factors',
            'Examine coaching impact on team success'
        ]
    },
    'coaches_integrated': {
        'dataset': coaches_integrated,
        'description': 'Coach data merged with team performance and championship results',
        'key_use_cases': [
            'Evaluate coaching effectiveness',
            'Compare coach performance across teams',
            'Study championship-winning coaches'
        ]
    },
    'awards_integrated': {
        'dataset': awards_integrated,
        'description': 'Award winners merged with player stats, team info, and championships',
        'key_use_cases': [
            'Analyze award winner characteristics',
            'Study correlation between awards and championships',
            'Compare performance of award winners'
        ]
    }
}

for name, info in datasets_info.items():
    print(f"{name.upper()}")
    print(f"   Records: {len(info['dataset']):,}")
    print(f"   Columns: {info['dataset'].shape[1]}")
    print(f"   Description: {info['description']}")
    #print(f"   Key Use Cases:")
    for use_case in info['key_use_cases']:
        print(f"      • {use_case}")
    print()

print("\nAll datasets have been successfully integrated and are ready for analysis and modeling.")